# 02 – Régression logistique (modèle de référence)

**Objectif :** établir la *baseline* du projet. Les autres modèles (Random Forest, XGBoost, MLP) devront faire mieux.

Pourquoi commencer par la régression logistique :
- c'est le **standard historique du credit scoring** (grilles de score bancaires) ;
- elle produit directement une **probabilité de défaut** ;
- ses coefficients sont **interprétables** (odds ratios), ce que les régulateurs apprécient.

**Plan du notebook**
1. Chargement et prétraitement (`src/preprocessing.py`)
2. Baseline brute (sans correction du déséquilibre)
3. Correction du déséquilibre (`class_weight="balanced"`)
4. Réglage des hyperparamètres (C, L1/L2) par validation croisée stratifiée
5. Choix du seuil de décision (sur le train, en out-of-fold)
6. Évaluation finale sur le test : AUC, F1, matrice de confusion, calibration
7. Interprétation : odds ratios
8. Variante sans variables sensibles (SEX, AGE, MARRIAGE)

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_score,
                             recall_score, accuracy_score, brier_score_loss, confusion_matrix,
                             roc_curve, precision_recall_curve, ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve

# --- Localisation automatique du projet -------------------------------------
# Fonctionne avec la structure du dépôt (notebooks/, src/, data/) OU si tous les
# fichiers sont dans le même dossier.
ROOT = Path.cwd().parent if any((Path.cwd().parent / d).exists() for d in ("src", "data")) else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(Path.cwd()))

def find_data():
    # Le zip UCI contient « default of credit card clients.xls » (avec des espaces)
    for folder in [ROOT / "data", ROOT, Path.cwd()]:
        for pattern in ["default*credit*card*.xls", "default*credit*card*.xlsx", "default*credit*card*.csv"]:
            hits = sorted(folder.glob(pattern))
            if hits:
                return hits[0]
    raise FileNotFoundError(
        f"Fichier de données introuvable dans {ROOT / 'data'} ni dans {Path.cwd()}.\n"
        "Téléchargez-le sur https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients "
        "puis placez le .xls dans le dossier data/ (ou à côté du notebook).")

from preprocessing import (load_data, clean_data, add_features, split, build_preprocessor,
                           NOMINAL, TARGET, RANDOM_STATE)

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
RES_DIR = ROOT / "results"
FIG_DIR = RES_DIR / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = find_data()
print("scikit-learn", sklearn.__version__)
print("Données :", DATA_PATH)

## 1. Données

On réutilise exactement le pipeline de la section 6 du README : nettoyage, variables construites, split 80/20 stratifié.
Le **test n'est plus touché** jusqu'à l'évaluation finale (section 6 de ce notebook).

In [ ]:
df = add_features(clean_data(load_data(DATA_PATH)))
X_train, X_test, y_train, y_test = split(df)
numeric_cols = [c for c in X_train.columns if c not in NOMINAL]

print(f"Train : {X_train.shape} | taux de défaut = {y_train.mean():.3f}")
print(f"Test  : {X_test.shape} | taux de défaut = {y_test.mean():.3f}")

### Outils communs

- `make_logreg` : crée une régression logistique L1 ou L2, compatible avec toutes les versions de scikit-learn (l'argument `penalty` est déprécié depuis la 1.8).
- `make_pipeline` : prétraitement (one-hot, log, standardisation) **+** modèle dans un même `Pipeline`. Le scaler est donc ré-ajusté dans chaque fold de validation croisée : aucune fuite de données.
- `evaluate` : calcule toutes les métriques du projet à partir des probabilités et d'un seuil.

In [ ]:
SK_NEW = tuple(int(v) for v in sklearn.__version__.split(".")[:2]) >= (1, 8)

def make_logreg(C=1.0, l1=False, class_weight=None):
    kw = dict(C=C, solver="liblinear", class_weight=class_weight, max_iter=5000, random_state=RANDOM_STATE)
    if SK_NEW:
        kw["l1_ratio"] = 1.0 if l1 else 0.0
    else:
        kw["penalty"] = "l1" if l1 else "l2"
    return LogisticRegression(**kw)

def make_pipeline(clf, num_cols=numeric_cols):
    return Pipeline([("pre", build_preprocessor(num_cols)), ("clf", clf)])

def evaluate(name, y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {"model": name, "threshold": round(threshold, 3),
            "ROC_AUC": roc_auc_score(y_true, proba),
            "PR_AUC": average_precision_score(y_true, proba),
            "F1": f1_score(y_true, pred), "Precision": precision_score(y_true, pred),
            "Recall": recall_score(y_true, pred), "Accuracy": accuracy_score(y_true, pred),
            "Brier": brier_score_loss(y_true, proba)}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []

## 2. Baseline brute

Régression logistique L2 par défaut (C = 1), sans correction du déséquilibre, seuil 0,5.
On s'attend à une bonne accuracy mais à un **rappel faible** : le modèle « joue la sécurité » en prédisant surtout la classe majoritaire.

In [ ]:
base = make_pipeline(make_logreg()).fit(X_train, y_train)
p_base = base.predict_proba(X_test)[:, 1]
results.append(evaluate("LR baseline (seuil 0,5)", y_test, p_base))
pd.DataFrame(results).round(3)

## 3. Correction du déséquilibre : `class_weight="balanced"`

Chaque défaut pèse environ 3,5 fois plus qu'un non-défaut dans la fonction de perte (ratio 77,9 / 22,1).
Le modèle est alors pénalisé davantage quand il rate un défaut.

In [ ]:
bal = make_pipeline(make_logreg(class_weight="balanced")).fit(X_train, y_train)
p_bal = bal.predict_proba(X_test)[:, 1]
results.append(evaluate("LR balanced (seuil 0,5)", y_test, p_bal))
pd.DataFrame(results).round(3)

**À retenir :** l'AUC change très peu. La pondération ne modifie presque pas le **classement** des clients, elle **décale les probabilités vers le haut**.
Le rappel monte fortement, la précision baisse : c'est surtout un changement de seuil implicite.
Contrepartie : les probabilités ne sont plus de vraies PD (Brier score moins bon). On le vérifie en section 6.

## 4. Réglage des hyperparamètres

Grille sur :
- **C** (inverse de la régularisation) : de 0,001 (forte régularisation) à 100 ;
- **type de pénalité** : L2 (réduit tous les coefficients) ou L1 (en met certains à zéro : sélection de variables, utile vu la forte colinéarité des `BILL_AMT`).

Critère : **ROC-AUC** en validation croisée stratifiée à 5 plis, sur le train uniquement.

In [ ]:
C_grid = np.logspace(-3, 2, 6)
param_grid = {"clf": [make_logreg(C=C, l1=l1, class_weight="balanced") for C in C_grid for l1 in (False, True)]}

grid = GridSearchCV(make_pipeline(make_logreg()), param_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
grid.fit(X_train, y_train)

cv_res = pd.DataFrame({
    "C": [p["clf"].C for p in grid.cv_results_["params"]],
    "penalty": ["L1" if (getattr(p["clf"], "l1_ratio", 0) == 1 or getattr(p["clf"], "penalty", "") == "l1") else "L2"
                for p in grid.cv_results_["params"]],
    "AUC_cv_mean": grid.cv_results_["mean_test_score"],
    "AUC_cv_std": grid.cv_results_["std_test_score"]})
best_clf = grid.best_params_["clf"]
print("Meilleur modèle :", best_clf)
print(f"AUC CV = {grid.best_score_:.4f}")

fig, ax = plt.subplots(figsize=(6, 3.5))
for pen, g in cv_res.groupby("penalty"):
    ax.errorbar(g["C"], g["AUC_cv_mean"], yerr=g["AUC_cv_std"], marker="o", capsize=3, label=pen)
ax.set_xscale("log"); ax.set_xlabel("C (log)"); ax.set_ylabel("ROC-AUC (CV 5 plis)")
ax.set_title("Réglage de C et de la pénalité"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "logreg_tuning.png", dpi=150); plt.show()
cv_res.sort_values("AUC_cv_mean", ascending=False).head().round(4)

## 5. Choix du seuil de décision

Le seuil 0,5 n'a rien de naturel ici. On choisit le seuil qui **maximise le F1** de la classe « défaut ».

Pour éviter de « tricher » avec le test, le seuil est choisi sur des **probabilités out-of-fold du train** (`cross_val_predict`) : chaque client du train est prédit par un modèle qui ne l'a pas vu.

In [ ]:
best_pipe = make_pipeline(best_clf)
oof = cross_val_predict(best_pipe, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

prec, rec, thr = precision_recall_curve(y_train, oof)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_thr = float(thr[np.argmax(f1s)])
print(f"Seuil optimal (F1, out-of-fold) = {best_thr:.3f} | F1 OOF = {f1s.max():.3f}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(thr, prec[:-1], label="Précision"); ax.plot(thr, rec[:-1], label="Rappel"); ax.plot(thr, f1s, label="F1", lw=2)
ax.axvline(best_thr, ls="--", c="grey"); ax.set_xlabel("Seuil"); ax.set_title("Choix du seuil (train, out-of-fold)"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "logreg_threshold.png", dpi=150); plt.show()

## 6. Évaluation finale sur le test

On compare les variantes sur les mêmes 5 993 clients jamais vus. À lire : la ligne « LR tunée (seuil optimisé) » est le modèle de référence retenu pour la suite.

In [ ]:
best_pipe.fit(X_train, y_train)
p_best = best_pipe.predict_proba(X_test)[:, 1]
results.append(evaluate("LR tunée (seuil 0,5)", y_test, p_best))
results.append(evaluate("LR tunée (seuil optimisé)", y_test, p_best, best_thr))
res_df = pd.DataFrame(results).set_index("model").round(3)
res_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# ROC
for name, p in [("Baseline", p_base), ("Tunée (balanced)", p_best)]:
    fpr, tpr, _ = roc_curve(y_test, p)
    axes[0].plot(fpr, tpr, label=f"{name} – AUC {roc_auc_score(y_test, p):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1); axes[0].set_xlabel("Taux de faux positifs"); axes[0].set_ylabel("Taux de vrais positifs")
axes[0].set_title("Courbe ROC (test)"); axes[0].legend(loc="lower right")

# Matrice de confusion au seuil optimisé
cm = confusion_matrix(y_test, (p_best >= best_thr).astype(int))
ConfusionMatrixDisplay(cm, display_labels=["Pas de défaut", "Défaut"]).plot(ax=axes[1], cmap="Blues", colorbar=False)
axes[1].set_title(f"Matrice de confusion (seuil {best_thr:.2f})"); axes[1].grid(False)
axes[1].set_xlabel("Prédiction"); axes[1].set_ylabel("Réalité")

# Calibration
for name, p in [("Baseline (non pondérée)", p_base), ("Tunée (balanced)", p_best)]:
    frac, mean_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    axes[2].plot(mean_pred, frac, marker="o", label=name)
axes[2].plot([0, 1], [0, 1], "k--", lw=1); axes[2].set_xlabel("PD prédite"); axes[2].set_ylabel("Taux de défaut observé")
axes[2].set_title("Calibration (test)"); axes[2].legend()

plt.tight_layout(); plt.savefig(FIG_DIR / "logreg_evaluation.png", dpi=150); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Défauts détectés : {tp}/{tp+fn} ({tp/(tp+fn):.0%}) | défauts manqués : {fn} | fausses alertes : {fp}")

**Lecture de la calibration :** la courbe du modèle pondéré est **sous la diagonale**. Il surestime la PD (par construction, à cause de `class_weight`).
Pour classer les clients et prendre une décision avec un seuil, ce n'est pas un problème. En revanche, si la PD sert à calculer une perte attendue (PD × LGD × EAD), il faut utiliser le modèle non pondéré ou **recalibrer** (`CalibratedClassifierCV`).

## 7. Interprétation : odds ratios

Comment lire $e^{\beta}$ (l'odds ratio) :
- **variables numériques (standardisées)** : une hausse d'**un écart-type** multiplie la cote du défaut par $e^{\beta}$ ;
- **indicatrices one-hot** (`SEX_2`, `EDUCATION_x`, `MARRIAGE_x`) : $e^{\beta}$ compare la catégorie à la **catégorie de référence** (homme, master/doctorat, marié). Elles sont en 0/1 et non standardisées, donc leur coefficient n'est **pas directement comparable** en taille à celui d'une variable numérique ;
- $e^{\beta} > 1$ : augmente le risque ; $e^{\beta} < 1$ : le réduit ; $\beta = 0$ : variable écartée par la pénalité L1.

⚠️ Les `BILL_AMT` et `PAY_x` sont très corrélés entre eux : leurs coefficients individuels sont instables (certains `PAY_x` ressortent même négatifs, parce que `N_MONTHS_LATE` et `MAX_DELAY` captent déjà l'information). Il faut lire les **familles de variables** plutôt que chaque coefficient isolé.

In [ ]:
feat_names = best_pipe.named_steps["pre"].get_feature_names_out()
feat_names = [f.split("__", 1)[1] for f in feat_names]
coefs = pd.DataFrame({"feature": feat_names, "coef": best_pipe.named_steps["clf"].coef_[0]})
coefs["odds_ratio"] = np.exp(coefs["coef"])
coefs = coefs.reindex(coefs["coef"].abs().sort_values(ascending=False).index)
print(f"Variables mises à zéro : {(coefs['coef'] == 0).sum()} / {len(coefs)}")

top = coefs.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top["feature"], top["coef"], color=np.where(top["coef"] > 0, "#c0392b", "#2e86c1"))
ax.axvline(0, c="k", lw=0.8); ax.set_xlabel("Coefficient (variables standardisées)")
ax.set_title("Régression logistique : 15 coefficients les plus forts\n(rouge = augmente le risque, bleu = le réduit)")
plt.tight_layout(); plt.savefig(FIG_DIR / "logreg_coefficients.png", dpi=150); plt.show()
coefs.head(15).round(3).reset_index(drop=True)

**Lecture économique**
- Les variables les plus fortes sont celles du **comportement de retard** (`N_MONTHS_LATE`, `MAX_DELAY`, `PAY_1`) : un client qui a déjà été en retard, et surtout récemment, est beaucoup plus risqué. C'est cohérent avec la logique bancaire : le passé de paiement est le meilleur prédicteur du défaut.
- Un **plafond élevé** (`LIMIT_BAL`) et de **gros remboursements** (`PAY_AMT`) réduisent le risque : la banque accorde plus aux clients solvables, et ceux qui remboursent beaucoup ont de la marge.
- Un **taux d'utilisation élevé** (`UTIL_MEAN`) augmente le risque : le client vit au bord de son plafond.
- `EDUCATION_4` a le coefficient le plus fort, mais c'est **un petit groupe** (468 clients, 7 % de défauts contre 22 % en moyenne) formé en partie de codes non documentés regroupés. On ne doit pas lui donner trop de poids dans l'interprétation.

In [ ]:
df.groupby("EDUCATION")[TARGET].agg(taux_defaut="mean", effectif="size").round(3)

## 8. Variante sans variables sensibles

`SEX`, `AGE` et `MARRIAGE` sont des caractéristiques protégées : les utiliser pour une décision de crédit pose un problème de discrimination.
On ré-entraîne le même modèle sans elles pour mesurer ce qu'on « perd » à les retirer.

In [ ]:
from sklearn.base import clone
SENSITIVE = ["SEX", "AGE", "MARRIAGE"]

X_tr_fair, X_te_fair = X_train.drop(columns=SENSITIVE), X_test.drop(columns=SENSITIVE)
nom_fair = [c for c in NOMINAL if c not in SENSITIVE]          # il reste EDUCATION
num_fair = [c for c in X_tr_fair.columns if c not in nom_fair]

fair_pipe = Pipeline([("pre", build_preprocessor(num_fair, nominal=nom_fair)),
                      ("clf", clone(best_clf))]).fit(X_tr_fair, y_train)

p_fair = fair_pipe.predict_proba(X_te_fair)[:, 1]
fair = evaluate("LR tunée sans SEX/AGE/MARRIAGE (seuil optimisé)", y_test, p_fair, best_thr)
results.append(fair)
print(f"AUC avec : {roc_auc_score(y_test, p_best):.4f} | sans : {fair['ROC_AUC']:.4f} | écart : {roc_auc_score(y_test, p_best) - fair['ROC_AUC']:+.4f}")

## 9. Sauvegarde des résultats

Les métriques sont enregistrées dans `results/metrics_logreg.csv` pour le tableau comparatif final avec Random Forest, XGBoost et MLP.

In [ ]:
final = pd.DataFrame(results).set_index("model").round(4)
final.to_csv(RES_DIR / "metrics_logreg.csv")
final